# TGIF-Train-Clean-Subset Cohort Acquisition Launcher (Colab CPU)

> **Phase**: Phase 4C.7B trace — Independent Cohort Acquisition (Existing Benchmark Intake)
> **Execution Environment**: Google Colab Standard CPU Runtime (0 GPU required, 0 VRAM)
> **Approved Scope**: INTAKE ONLY (Download 2 archives ~18.63 GiB, selective extract 400 pairs, package ~120-160 MB ZIP return to local. Detector, evaluation, and training NOT AUTHORIZED)
> **Approved Cohort**: $N=400$ pairs (14 Large [3.5%], 221 Medium [55.25%], 165 Small [41.25%])
> **Locked Manifest SHA-256**: `53a6ee472fe840a42abd97ccb7475932e0720f5788f745f57f7a2bcfbc32cc8c`

---

In [ ]:
# Cell 1: Environment & Disk Capacity Preflight
import os
import shutil
import sys

print("=== Cell 1: Environment Preflight ===")
print(f"Python Version: {sys.version}")
if sys.version_info < (3, 10):
    raise RuntimeError("Python 3.10+ required.")

total, used, free = shutil.disk_usage("/content")
free_gib = free / (1024 ** 3)
print(f"Colab /content Disk: Total = {total / (1024**3):.1f} GiB, Free = {free_gib:.1f} GiB")
if free_gib < 25.0:
    raise RuntimeError(f"Insufficient disk space ({free_gib:.1f} GiB < 25.0 GiB) to download 18.63 GiB archives.")

!pip install -q Pillow numpy
import numpy as np
import PIL
print(f"Pillow Version: {PIL.__version__}")
print(f"NumPy Version: {np.__version__}")
print("Environment preflight PASS.")

In [ ]:
# Cell 2: Repository Clone, Full-SHA Checkout & Locked Manifest Verification
import hashlib
from pathlib import Path
import subprocess

print("=== Cell 2: Repository Clone & Manifest Verification ===")
REPO_DIR = Path("/content/forensics-web-lab")
BRANCH = "research/independent-cohort-acquisition"
EXPECTED_EXECUTION_COMMIT = "05b824d671577dd2f71a24058048eee8be56e279"
EXPECTED_LOCKED_MANIFEST_SHA256 = "53a6ee472fe840a42abd97ccb7475932e0720f5788f745f57f7a2bcfbc32cc8c"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", "https://github.com/nomozer/forensics-web-lab.git", str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", EXPECTED_EXECUTION_COMMIT], check=True)
actual_commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if actual_commit != EXPECTED_EXECUTION_COMMIT:
    raise RuntimeError(f"Execution commit mismatch: {actual_commit} != {EXPECTED_EXECUTION_COMMIT}")
if subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain"], text=True).strip():
    raise RuntimeError("Execution checkout is not clean.")
print(f"Execution Commit: {actual_commit}")

manifest_path = REPO_DIR / "research/evidence/phase-4c.7b/tgif_train_clean_subset_manifest_locked_n400.json"
if not manifest_path.is_file():
    raise FileNotFoundError(f"Manifest not found: {manifest_path}")

manifest_hasher = hashlib.sha256()
with open(manifest_path, "rb") as f:
    while chunk := f.read(1024 * 1024):
        manifest_hasher.update(chunk)
actual_sha = manifest_hasher.hexdigest()

print(f"Manifest Path: {manifest_path}")
print(f"Actual SHA-256: {actual_sha}")
if actual_sha != EXPECTED_LOCKED_MANIFEST_SHA256:
    raise RuntimeError(f"Manifest SHA mismatch: {actual_sha} != {EXPECTED_LOCKED_MANIFEST_SHA256}")
print("Locked Manifest Verification PASS: Exact hash match.")

In [ ]:
# Cell 3: Download TGIF Training Archives from Official Nextcloud
sys.path.insert(0, str(REPO_DIR))
from scripts.research.acquire_tgif_train_subset_colab import (
    ARCHIVE_BUDGET,
    ARCHIVE_URLS,
    EXPECTED_LOCKED_MANIFEST_SHA256 as WORKER_MANIFEST_SHA256,
    TGIF_NEXTCLOUD_BASE,
    download_file_with_progress,
)

print("=== Cell 3: Download TGIF Archives (18.63 GiB) ===")
if WORKER_MANIFEST_SHA256 != EXPECTED_LOCKED_MANIFEST_SHA256:
    raise RuntimeError("Notebook/worker locked-manifest binding mismatch.")
if TGIF_NEXTCLOUD_BASE != "https://cloud.ilabt.imec.be/index.php/s/xEeAzrY7ES9KA8o":
    raise RuntimeError("Unexpected TGIF source endpoint.")
ARCHIVES = (
    ("orig_training.tar.gz", ARCHIVE_URLS["orig_training"], ARCHIVE_BUDGET["orig_training_tar_gz_bytes"]),
    ("sd2-sp_training.tar.gz", ARCHIVE_URLS["sd2_training"], ARCHIVE_BUDGET["sd2_training_tar_gz_bytes"]),
)

content_dir = Path("/content")
for filename, url, expected_bytes in ARCHIVES:
    download_file_with_progress(url, content_dir / filename, expected_bytes)

print("All archives downloaded and size verified.")

In [ ]:
# Cell 4: Selective Stream Extraction & Package Creation
import subprocess

print("=== Cell 4: Execute Selective Extraction Worker ===")
cmd = [
    sys.executable,
    str(REPO_DIR / "scripts/research/acquire_tgif_train_subset_colab.py"),
    "--manifest", str(manifest_path),
    "--archive-dir", "/content",
    "--output-zip", "/content/tgif_train_clean_subset_package.zip",
]

res = subprocess.run(cmd, capture_output=False, text=True)
if res.returncode != 0:
    raise RuntimeError(f"Intake worker failed with exit code {res.returncode}.")
print("Worker execution PASS: Package zip generated.")

In [ ]:
# Cell 5: Package Verification & Handover Instructions
import json
from google.colab import files

print("=== Cell 5: Package Handover ===")
package_zip = Path("/content/tgif_train_clean_subset_package.zip")
if not package_zip.is_file():
    raise FileNotFoundError(f"Package not found: {package_zip}")

zip_bytes = package_zip.stat().st_size
zip_sha = sha256_file(package_zip)

print(f"Package File: {package_zip.name}")
print(f"Size: {zip_bytes / (1024 * 1024):.2f} MB ({zip_bytes:,} bytes)")
print(f"SHA-256: {zip_sha}")

print("\n--- NEXT STEPS ON LOCAL WORKSTATION ---")
print("1. Download 'tgif_train_clean_subset_package.zip' to local workstation")
print("   (into folder: data/research/local-artifacts/phase-4c.7b/)")
print("2. Run local forensic audit and tripartite verification:")
print(r"   .\ml\.venv\Scripts\python.exe scripts/research/audit_tgif_train_subset_local.py --package-zip data/research/local-artifacts/phase-4c.7b/tgif_train_clean_subset_package.zip")
print("3. Stop before evaluation to inspect the generated contact sheet.")

# Trigger browser download in Colab, then remove only ephemeral archives/partials.
files.download(str(package_zip))
for archive_name, _, _ in ARCHIVES:
    (Path("/content") / archive_name).unlink(missing_ok=True)
    (Path("/content") / f"{archive_name}.part").unlink(missing_ok=True)
print(f"Colab cleanup complete. ZIP retained at: {package_zip}")